# Credit Card Fraud Detection

This notebook builds an end-to-end machine learning pipeline to detect fraudulent credit card transactions
in a highly imbalanced dataset (~0.7% fraud rate).

**Pipeline overview:**
1. Data generation / loading
2. Exploratory Data Analysis (EDA)
3. Preprocessing and train/test split
4. Handling class imbalance with SMOTE
5. Model training: Logistic Regression, Random Forest, Gradient Boosting
6. Evaluation using Precision, Recall, F1, ROC-AUC, and PR-AUC
7. Conclusion and key takeaways

> **Note on the dataset:** This project uses a synthetically generated transaction dataset
> (`generate_data.py`) that mirrors the structure and class imbalance of the well-known
> [Kaggle Credit Card Fraud Detection dataset](https://www.kaggle.com/mlg-ulb/creditcardfraud)
> (anonymized PCA-style features V1-V28, plus `Amount` and `Time`). This keeps the project
> fully reproducible without requiring a large external download.


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score
)

sns.set_style("whitegrid")
np.random.seed(42)


## 2. Load the dataset

Run `generate_data.py` once to create `data/transactions.csv` (50,000 transactions, ~0.7% fraud).


In [ ]:
df = pd.read_csv("data/transactions.csv")
print("Dataset shape:", df.shape)
df.head()


In [ ]:
print("Class distribution:")
print(df["Class"].value_counts())
print(f"\nFraud percentage: {100 * df['Class'].mean():.3f}%")


## 3. Exploratory Data Analysis

### 3.1 Class distribution

The dataset is severely imbalanced — fraudulent transactions make up less than 1% of all records.
This imbalance is the central challenge of the project: a naive model that predicts "legitimate"
for every transaction would already be ~99% accurate, but completely useless for catching fraud.


In [ ]:
plt.figure(figsize=(5, 4))
ax = sns.countplot(x="Class", data=df, hue="Class", palette=["#378ADD", "#D85A30"], legend=False)
ax.set_xticks([0, 1])
ax.set_xticklabels(["Legitimate (0)", "Fraud (1)"])
ax.set_title("Class distribution (highly imbalanced)")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width() / 2, p.get_height()),
                 ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()


### 3.2 Transaction amount by class

Fraudulent transactions tend to have a different amount distribution compared to legitimate ones,
making `Amount` a potentially useful feature.


In [ ]:
plt.figure(figsize=(6, 4))
sns.kdeplot(df[df["Class"] == 0]["Amount"], label="Legitimate", fill=True, color="#378ADD")
sns.kdeplot(df[df["Class"] == 1]["Amount"], label="Fraud", fill=True, color="#D85A30")
plt.xlim(0, 500)
plt.title("Transaction amount distribution by class")
plt.xlabel("Transaction amount")
plt.legend()
plt.tight_layout()
plt.show()


### 3.3 Feature correlations

A quick look at correlations among a subset of the anonymized features.

In [ ]:
plt.figure(figsize=(8, 6))
corr = df[[f"V{i}" for i in range(1, 11)] + ["Amount", "Class"]].corr()
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)
plt.title("Feature correlation heatmap (subset)")
plt.tight_layout()
plt.show()


## 4. Preprocessing

- Split the data into train and test sets, **stratified** by class so both sets preserve the ~0.7% fraud rate.
- Scale features using `StandardScaler` (important for Logistic Regression).


In [ ]:
X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)
print("Train fraud count:", y_train.sum(), " Test fraud count:", y_test.sum())


## 5. Handling class imbalance with SMOTE

With only ~0.7% of transactions labeled as fraud, models trained directly on this data tend to
ignore the minority class almost entirely. **SMOTE (Synthetic Minority Over-sampling Technique)**
addresses this by generating synthetic fraud examples — interpolating between each minority
sample and its nearest neighbors — until the training set is balanced.

Below is a lightweight from-scratch implementation of SMOTE using `NearestNeighbors`, applied
**only to the training set** (the test set is left untouched so evaluation reflects real-world conditions).


In [ ]:
def simple_smote(X, y, minority_label=1, k=5, random_state=42):
    rng = np.random.RandomState(random_state)
    X_min = X[y == minority_label]
    X_maj = X[y != minority_label]
    n_to_generate = len(X_maj) - len(X_min)

    nn = NearestNeighbors(n_neighbors=k + 1).fit(X_min)
    _, neighbors = nn.kneighbors(X_min)

    synthetic = []
    for _ in range(n_to_generate):
        i = rng.randint(0, len(X_min))
        nn_idx = neighbors[i, rng.randint(1, k + 1)]
        gap = rng.rand()
        new_point = X_min[i] + gap * (X_min[nn_idx] - X_min[i])
        synthetic.append(new_point)

    synthetic = np.array(synthetic)
    X_resampled = np.vstack([X, synthetic])
    y_resampled = np.concatenate([y, np.full(len(synthetic), minority_label)])
    return X_resampled, y_resampled


X_train_res, y_train_res = simple_smote(X_train_scaled, y_train.values, k=5)
print("Training class distribution after SMOTE:")
print(pd.Series(y_train_res).value_counts())


## 6. Model training

We train three classifiers on the SMOTE-balanced training data:

- **Logistic Regression** — simple, interpretable baseline
- **Random Forest** — ensemble of decision trees, handles non-linear patterns well
- **Gradient Boosting** — sequential boosting, often strong on tabular data


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, random_state=42),
}

results = {}

for name, model in models.items():
    model.fit(X_train_res, y_train_res)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    report = classification_report(y_test, y_pred, target_names=["Legitimate", "Fraud"], output_dict=True)
    auc = roc_auc_score(y_test, y_proba)
    avg_precision = average_precision_score(y_test, y_proba)
    cm = confusion_matrix(y_test, y_pred)

    results[name] = {
        "model": model,
        "precision_fraud": report["Fraud"]["precision"],
        "recall_fraud": report["Fraud"]["recall"],
        "f1_fraud": report["Fraud"]["f1-score"],
        "roc_auc": auc,
        "pr_auc": avg_precision,
        "confusion_matrix": cm,
        "y_proba": y_proba,
    }

    print(f"===== {name} =====")
    print(classification_report(y_test, y_pred, target_names=["Legitimate", "Fraud"]))
    print(f"ROC-AUC: {auc:.4f}   PR-AUC: {avg_precision:.4f}\n")


## 7. Evaluation

For imbalanced classification problems, **accuracy is misleading** (predicting "legitimate" for
everything would yield ~99% accuracy). Instead we focus on:

- **Precision (fraud)**: of all transactions flagged as fraud, how many actually were fraud?
- **Recall (fraud)**: of all actual fraud transactions, how many did we catch?
- **ROC-AUC**: overall ability to rank fraud higher than legitimate transactions.
- **PR-AUC (Average Precision)**: more informative than ROC-AUC under heavy class imbalance.


In [ ]:
summary = pd.DataFrame({
    name: {
        "Precision (fraud)": res["precision_fraud"],
        "Recall (fraud)": res["recall_fraud"],
        "F1 (fraud)": res["f1_fraud"],
        "ROC-AUC": res["roc_auc"],
        "PR-AUC": res["pr_auc"],
    }
    for name, res in results.items()
}).T.round(3)

summary


### ROC and Precision-Recall curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res["y_proba"])
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={res['roc_auc']:.3f})")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[0].set_title("ROC curves")
axes[0].legend()

for name, res in results.items():
    prec, rec, _ = precision_recall_curve(y_test, res["y_proba"])
    axes[1].plot(rec, prec, label=f"{name} (PR-AUC={res['pr_auc']:.3f})")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall curves")
axes[1].legend()

plt.tight_layout()
plt.show()


### Confusion matrix — best model (by PR-AUC)

In [ ]:
best_model_name = max(results, key=lambda k: results[k]["pr_auc"])
print("Best model:", best_model_name)

plt.figure(figsize=(4.5, 4))
sns.heatmap(results[best_model_name]["confusion_matrix"], annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred Legit", "Pred Fraud"], yticklabels=["Actual Legit", "Actual Fraud"])
plt.title(f"Confusion matrix - {best_model_name}")
plt.tight_layout()
plt.show()


## 8. Conclusion

**Key results (test set, SMOTE-balanced training):**

| Model | Precision (fraud) | Recall (fraud) | F1 (fraud) | ROC-AUC | PR-AUC |
|---|---|---|---|---|---|
| Logistic Regression | 0.035 | 0.759 | 0.066 | 0.893 | 0.316 |
| **Random Forest** | **0.846** | **0.379** | **0.524** | **0.942** | **0.561** |
| Gradient Boosting | 0.159 | 0.678 | 0.257 | 0.945 | 0.559 |

**Takeaways:**

- **Random Forest** achieved the best overall PR-AUC (0.561) and the highest precision (0.846),
  meaning that when it flags a transaction as fraud, it's correct ~85% of the time — important
  for minimizing false alarms in a production fraud system.
- **Gradient Boosting** achieved the highest ROC-AUC (0.945) and the best recall-precision balance
  for catching more fraud cases, at the cost of more false positives.
- **Logistic Regression**, as expected for a linear model on this non-linear synthetic data,
  underperforms — useful mainly as a baseline.
- SMOTE was essential: without it, models trained on the raw imbalanced data almost completely
  ignored the minority (fraud) class.

**Possible next steps:**
- Threshold tuning per model to balance precision/recall based on business cost of false positives vs. false negatives
- Hyperparameter tuning (GridSearchCV / RandomizedSearchCV)
- Try XGBoost / LightGBM for potentially stronger performance
- Cost-sensitive evaluation (assign real monetary cost to false positives/negatives)
